In [ ]:
from parflow import Run
from parflow.tools.fs import get_absolute_path
from parflow.tools.io import write_pfb, read_pfb
import numpy as np
import pandas as pd

# Create numpy array
topodata = pd.read_excel("./input/topology.xlsx")
# mpk = input_file.loc[:,['x_coord', 'z_coord']].to_numpy() 

In [ ]:
import shapely
from shapely import Polygon

topopoly = {}
for id in np.unique(topodata.id):
    data = topodata[topodata.id == id]
    topopoly[id] = Polygon([(xxx, zzz) for xxx, zzz in zip(data.x_coord, data.z_coord)])
print(topopoly[id])

In [ ]:
from modules import find_cells_within_polygon

NX = 100
NY = 3
NZ = 60

l_z = 29.8 # cm
l_x = 51.2 # cm
l_y = 2.54 # cm

DX = l_x/NX
DY = l_y/NY
DZ = l_z/NZ

x_cell_centers = np.arange(DX/2, l_x, DX)
z_cell_centers = np.arange(DZ/2, l_z, DZ)

gridx, gridz = np.meshgrid(x_cell_centers, z_cell_centers)

indicator = np.ones((NZ, NY, NX))*-999
holder = np.ones((NZ, NX))*-888

# defines geology of cross-section
for n, id in enumerate(topopoly.keys()):
    idx = find_cells_within_polygon(topopoly[id], gridx, gridz)
    n+=1
    if id == 'inactive':
        n = 0
    holder[idx] = n
    print(n, id)

# propagates cross-section through the model "rows"
for y in range(NY):
    indicator[:,y,:] = holder

In [ ]:
import matplotlib.pyplot as plt
for y in range(NY):
    plt.figure(y)
    plt.imshow(np.flipud(indicator[:, y,:]))
    plt.title(f'Layer {y}')
    plt.show()

In [ ]:

# Write flow boundary file as PFB with write_pfb() function
write_pfb(get_absolute_path('contamination_model.pfb'), indicator)

In [ ]:
## Parflow imports
from parflowio.pyParflowio import PFData


model_indicator = PFData('./contamination_model.pfb')
model_indicator.loadHeader()
model_indicator.loadData()
model_indicator_data = model_indicator.viewDataArray()

print(f'Dimensions of output file: {model_indicator_data.shape}')



In [ ]:
import matplotlib.pyplot as plt
for l in range(1):
    plt.figure(l)
    plt.imshow(model_indicator_data[:, l,:])
    plt.title(f'Layer {l}')
    plt.show()